# MASKED text — frozen rules, shared FULL base

Produce 114 MASKED opinions from the existing `full_v1` text using `masker_v1`.
The IDs, labels, families, and chronological assignments remain unchanged: 81
training, 16 development, 16 test, and one pending Sodhi case. Pending text is
exported for completeness; it is not assigned to an experiment.

Rules were developed on the 81 training texts, with complete reading of 10
purposively selected training opinions and automated screening of all 81.
`preprocessing/masked_training_review_v1.json` records the scope and audit.
The code, inputs, and training review were hashed in `masked_rules_v1.json`
before the first evaluation masking run. No label or selected annotation evidence
is passed to the masker. This is automated preprocessing with assistant review,
not human-adjudicated text or a guarantee that every direct cue is removed.


In [1]:
from pathlib import Path
import importlib.util
import json
import sys
sys.dont_write_bytecode = True
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "annotation/source/cases.jsonl").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Run inside legal-judgment.")
MVP = ROOT / "annotation/submissions/becca/MVP"
sys.path.insert(0, str(MVP / "preprocessing"))
import masked_text as masker
import verify_masked as verifier
print("Boundary and leakage regression checks:", verifier.check_rules())


Boundary and leakage regression checks: 20


## Apply frozen rules

Remove formal dispositions and explicit current-result sentences, including
repetitions. Preserve factual/procedural narration, cited outcomes, party requests,
and indirect reasoning. The same rules apply to every case. Whole-sentence
removal can also remove facts or reasoning within an outcome sentence.

Offsets in the audit refer to frozen FULL text, not raw source text. Each removal
includes its exact text and reason. Word counts use whitespace-separated words;
encoder token/chunk statistics await Yinkui's tokenizer and chunk settings.
Existing versioned exports cannot be overwritten with different results.


In [2]:
summary, audits = masker.build(ROOT)
print(json.dumps(summary, indent=2))
print("MASKED word count range:", min(a["masked_words"] for a in audits),
      "to", max(a["masked_words"] for a in audits))


{
  "version": "masked_v1",
  "case_count": 114,
  "partition_counts": {
    "test": 16,
    "train": 81,
    "pending_or_excluded": 1,
    "development": 16
  },
  "full_words": 63619,
  "masked_words": 58441,
  "removed_sentences": 223,
  "removal_reasons": {
    "disposition_block": 92,
    "current_appellate_result": 24,
    "reviewed_ruling_result": 68,
    "operative_relief_result": 11,
    "disposition_synopsis": 23,
    "concluding_relief_result": 2,
    "express_relief_resolution": 2,
    "response_to_explicit_relief_argument": 1
  },
  "flag_counts": {
    "retained_cue_candidates_review": 42,
    "unbalanced_brackets_review_segmentation": 4,
    "over_half_words_removed_review": 1
  },
  "empty_case_ids": [],
  "near_empty_case_ids": [],
  "review_status": "automated masking with training sample assistant review; not human-reviewed",
  "labels_or_annotation_evidence_used": false,
  "frozen_ids_splits_and_labels_changed": false,
  "removed_words": 5178,
  "removed_word_fracti

## Verify exports and audit

Check 114 unique IDs, exact reconstruction from removal offsets, source/output
hashes, word accounting, and agreement between the JSONL and individual text
files. The model input has only `case_id` and `text`. Labels remain in the
unchanged frozen split manifest; join by ID for all model variants.

The rule checks cover both result directions, repeated outcomes, abbreviations,
citations, party arguments, historical appeals, and hypothetical outcomes.


In [3]:
print(json.dumps(verifier.check_exports(ROOT), indent=2))
print("Review candidates by partition:")
for partition in ("train", "development", "test", "pending_or_excluded"):
    group = [a for a in audits if a["partition"] == partition]
    print(partition, "cases:", len(group), "flagged:", sum(bool(a["flags"]) for a in group))
print("Cases with more than half the words removed:",
      [a["case_id"] for a in audits if a["removed_word_fraction"] > .5])


{
  "verified_cases": 114,
  "verified_output_files": 117
}
Review candidates by partition:
train cases: 81 flagged: 33
development cases: 16 flagged: 8
test cases: 16 flagged: 2
pending_or_excluded cases: 1 flagged: 1
Cases with more than half the words removed: ['NY-5936535']


## Review limits and next use

Retained-cue flags are broad candidates: many are quoted precedent, procedural
history, or indirect reasoning. Bracket flags identify inherited OCR/formatting
issues. No evaluation-specific edits or rule tuning were performed. Evaluation
flags remain unresolved review candidates under the frozen rules.

Empty outputs retain their IDs and must be excluded from model fitting until
resolved in a new dataset version. Outputs under 50 words and removals over 50%
are flagged. The short training opinion NY-5936535 retains 64 words; its large
removal reflects a sentence combining the explicit result with reasoning.

This notebook does not train models, generate encoder chunks, revise labels,
resolve pending families, or promote provisional labels to human gold. Any later
rule or dataset change requires a new version and rerunning affected experiments.
